In [8]:
!pip install dask_ml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.0/150.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.4/259.4 kB 14.8 MB/s eta 0:00:00


In [15]:
import pandas as pd
import numpy as np
import sys
import joblib
from pathlib import Path

# --- CUML/RAPIDS IMPORTS ---
import cudf # GPU equivalent of pandas
import cupy as cp # GPU equivalent of numpy

# Use sklearn wrappers for GridSearch and Pipeline, but substitute the model
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

# Use CUML GPU estimator
from cuml.linear_model import LogisticRegression as CUML_LogisticRegression
# Removed: from cuml.preprocessing import StandardScaler as CUML_StandardScaler

from sklearn.metrics import classification_report, accuracy_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from dask_ml.model_selection import GridSearchCV as DaskGridSearchCV # Use Dask-ML's specialized GPU GridSearch
from dask.distributed import Client # For setting up a simple Dask client

In [16]:



OUT_DIR = ""

MODEL_OUT = "binary_logistic.joblib"

train_bc = pd.read_parquet('train_bc.parquet')
test_bc = pd.read_parquet('test_bc.parquet')
try:
    # Initialize a simple Dask client for local GPU parallelism
    client = Client(n_workers=1, threads_per_worker=1, memory_limit='8GB')
    print("Dask Client initialized for GPU parallelism.")
except Exception as e:
    print(f"Warning: Could not initialize Dask client. Training may fail or run on CPU: {e}")
# --- DATA CONVERSION TO CUDF (GPU) ---
# Convert data to cuDF for CUML estimators to run on GPU.
X_train = cudf.from_pandas(train_bc.drop('Attack', axis=1))
y_train = cudf.from_pandas(train_bc['Attack'].to_frame())['Attack']
X_test = cudf.from_pandas(test_bc.drop('Attack', axis=1))
y_test = cudf.from_pandas(test_bc['Attack'].to_frame())['Attack']
# ------------------------------------


print(f"Train data loaded (GPU): {len(X_train)} samples.")
print(f"Test data loaded (GPU): {len(X_test)} samples.")
print("-" * 30)

/usr/local/lib/python3.12/dist-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 45673 instead
  warnings.warn(
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:44295
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:45673/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:42175'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:44091 name: 0
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:44091
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:35376
INFO:distributed.scheduler:Receive client connection: Client-8b3d003c-cc55-11f0-81e4-0242ac1c000c
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:35390


Dask Client initialized for GPU parallelism.
Train data loaded (GPU): 509488 samples.
Test data loaded (GPU): 371272 samples.
------------------------------


In [19]:
# --- 1. Define Model Pipeline and EXPANDED Hyperparameter Grid ---

# Define the Pipeline: SCALER REMOVED. Only Logistic Regression remains.
logreg_cuml = CUML_LogisticRegression(
    max_iter=5000
)

# Define the EXPANDED Hyperparameter Grid: (KEPT IDENTICAL)
param_grid = [
    # Configuration 1: Solvers that support L1 regularization
    {
        'logreg__solver': ['liblinear', 'saga'],
        'logreg__penalty': ['l1', 'l2'],
        'logreg__C': np.logspace(-3, 3, 7)
    },
    # Configuration 2: Solvers that primarily support L2 regularization
    {
        'logreg__solver': ['lbfgs', 'sag'],
        'logreg__penalty': ['l2'],
        'logreg__C': np.logspace(-3, 3, 7)
    }
]

In [20]:
grid_search = DaskGridSearchCV(
    logreg_cuml,
    param_grid,
    cv=5,
    scoring='f1_macro', # Using F1-score for evaluation
    n_jobs=-1
)

print(f"Starting GPU-accelerated Grid Search across {len(param_grid[0]['logreg__C']) * (len(param_grid[0]['logreg__solver']) * len(param_grid[0]['logreg__penalty'])) + len(param_grid[1]['logreg__C']) * (len(param_grid[1]['logreg__solver']) * len(param_grid[1]['logreg__penalty']))} total parameter combinations...")
grid_search.fit(X_train, y_train)
print("Grid Search complete.")

Starting GPU-accelerated Grid Search across 42 total parameter combinations...


TypeError: Implicit conversion to a host NumPy array via __array__ is not allowed, To explicitly construct a GPU matrix, consider using .to_cupy()
To explicitly construct a host matrix, consider using .to_numpy().

In [ ]:
# --- 3. Evaluation and Print Logging ---

# 3a. Get Best Model and Params
best_logreg = grid_search.best_estimator_
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print("\n" + "="*50)
print("✨ GPU HYPERPARAMETER TUNING RESULTS (CUML) ✨")
print(f"Best cross-validation F1 score: {best_score:.4f}")
print(f"Best parameters found: {best_params}")
print("="*50)

# 3b. Evaluate on the Test Set
y_pred_cudf = best_logreg.predict(X_test)
y_pred_train_cudf = best_logreg.predict(X_train)

# --- CONVERT PREDICTIONS/TARGETS BACK TO NUMPY FOR SKLEARN METRICS/PLOTS ---
y_pred = y_pred_cudf.to_numpy()
y_pred_train = y_pred_train_cudf.to_numpy()
y_test_np = y_test.to_numpy()
y_train_np = y_train.to_numpy()
# -------------------------------------------------------------------------

joblib.dump(best_logreg, MODEL_OUT)

Starting GPU-accelerated Grid Search across 42 total parameter combinations...


TypeError: Implicit conversion to a host NumPy array via __array__ is not allowed, To explicitly construct a GPU matrix, consider using .to_cupy()
To explicitly construct a host matrix, consider using .to_numpy().

In [ ]:
# Calculate key metrics
test_accuracy = accuracy_score(y_test_np, y_pred)
test_f1 = f1_score(y_test_np, y_pred)

print("\n--- FINAL MODEL PERFORMANCE ON TEST SET ---")
print(f"Best Solver: {best_params['logreg__solver']}")
print(f"Test Set Accuracy: {test_accuracy:.4f}")
print(f"Test Set F1-Score: {test_f1:.4f}")


# Print full classification report
print("\nClassification Report (Test Set):\n")

test_cr = classification_report(y_test_np, y_pred)
print(test_cr)
print("-" * 50)

# Print full classification report
print("\nClassification Report (Train Set):\n")
train_cr = classification_report(y_train_np, y_pred_train)
print(train_cr)
print("-" * 50)

report_content = (
    "#" * 50 + "\n"
    "CLASSIFICATION REPORT (TRAIN SET)\n"
    "#" * 50 + "\n"
    f"{train_cr}\n\n"
    "#" * 50 + "\n"
    "CLASSIFICATION REPORT (TEST SET)\n"
    "#" * 50 + "\n"
    f"{test_cr}\n"
    )

REPORT_OUT_PATH = "classification_reports.txt"
with open(REPORT_OUT_PATH, 'w') as f:
  f.write(report_content)
print(f"\nSuccessfully dumped classification reports to: {REPORT_OUT_PATH}")


In [ ]:
def plot_classification_report(y_true, y_pred, title):
    """Plot classification report heatmap with support shown as plain numbers per row."""

    report_dict = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    df = pd.DataFrame(report_dict).transpose()

    # Extract support
    support = df['support'].fillna(0).astype(int)

    # Drop summary rows
    drop_rows = ['accuracy', 'macro avg', 'weighted avg', 'micro avg']
    df = df.drop(drop_rows, errors='ignore')

    # Keep only metric columns
    df_metrics = df.drop(columns=['support'], errors='ignore').astype(float)

    plt.figure(figsize=(8, 4))
    ax = sns.heatmap(
        df_metrics,
        annot=True,
        cmap="YlGnBu",
        fmt=".3f",
        linewidths=.5,
        linecolor='black',
        cbar=True
    )

    # Push the figure content slightly left so we have space on right
    plt.subplots_adjust(right=0.88)

    # Place support numbers farther right
    for y, cls in enumerate(df_metrics.index):
        sup_val = support.loc[cls]
        ax.text(
            df_metrics.shape[1] + 0.6,   # shifted right
            y + 0.5,
            str(sup_val),
            va='center',
            ha='left',
            fontsize=10,
            color='black'
        )

    # Support column header
    ax.text(
        df_metrics.shape[1] + 0.6,
        -0.2,
        "support",
        va='bottom',
        ha='left',
        fontsize=10,
        color='black',
        fontweight='bold'
    )

    plt.title(f"Classification Report Heatmap ({title})")
    plt.ylabel("Class")
    plt.xlabel("Metrics")
    plt.tight_layout()

    # Save the plot
    out_path =f"classification_report_{title.split()[0]}.png"
    plt.savefig(out_path)
    print(f"Saved {title} plot to: {out_path}")
    plt.show()


print("\n" + "="*50)
print("GENERATING CLASSIFICATION REPORT PLOTS")
print("="*50)

In [ ]:
# Plot 1: Train Set Classification Report
plot_classification_report(
    y_train_np,
    y_pred_train,
    'Train Set - Logistic Regression'
)

# Plot 2: Test Set Classification Report
plot_classification_report(
    y_test_np,
    y_pred,
    'Test Set - Logistic Regression'
)